In [ ]:
# Exact Riemann Problem Solver for Water Shock Tube
# Based on: R. Jishnu Chandran & A. Salih (2022)
# "Development of a benchmark solution in compressible liquid flows: 
# analytical solution to the water shock tube problem"
# Journal of Thermal Analysis and Calorimetry (2022) 147:5279–5292

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve

# ==============================================================================
# Modified NASG Equation of State Parameters for Water
# ==============================================================================

class ModifiedNASG:
    """Modified Noble-Abel Stiffened Gas EOS for water"""

    def __init__(self):
        # Parameters from the paper (Table 1)
        self.cp = 8006.79          # J/(kg·K)
        self.cv = 4130         # J/(kg·K)
        self.gamma = 1.93861        # Specific heat ratio
        self.P_inf = 1.15888e9    # Pa
        self.b = 0       # m³/kg
        self.q = 0      # J/kg (heat bond value)
        self.R_W = (self.gamma - 1) * self.cv  # Characteristic constant

    def specific_volume(self, P, T):
        """Calculate specific volume from pressure and temperature"""
        v = ((self.gamma - 1) * self.cv * T) / (P + self.P_inf) + self.b
        return v

    def density(self, P, T):
        """Calculate density from pressure and temperature"""
        v = self.specific_volume(P, T)
        rho = 1.0 / v
        return rho

    def stiffened_pressure(self, P):
        """Calculate stiffened pressure"""
        return P + self.P_inf

    def stiffened_density(self, rho):
        """Calculate stiffened density"""
        v = 1.0 / rho
        rho_bar = 1.0 / (v - self.b)
        return rho_bar

    def sound_speed(self, P, T):
        """Calculate speed of sound"""
        v = self.specific_volume(P, T)
        rho_bar = 1.0 / (v - self.b)
        P_bar = P + self.P_inf
        a_bar = np.sqrt(self.gamma * P_bar / rho_bar)
        return a_bar

    def internal_energy(self, P, v):
        """Calculate specific internal energy"""
        epsilon = ((P + self.gamma * self.P_inf) * (v - self.b)) / (self.gamma - 1) + self.q
        return epsilon

# ==============================================================================
# Exact Riemann Solver for Water Shock Tube
# ==============================================================================

class WaterShockTubeSolver:
    """Exact Riemann problem solver for water shock tube using Modified NASG EOS"""

    def __init__(self, P_L, P_R, T_L, T_R, x0, L):
        """
        Initialize water shock tube problem

        Parameters:
        -----------
        P_L : float
            Left chamber pressure (Pa)
        P_R : float
            Right chamber pressure (Pa)
        T_L : float
            Left chamber temperature (K)
        T_R : float
            Right chamber temperature (K)
        x0 : float
            Initial diaphragm position (m)
        L : float
            Total tube length (m)
        """
        self.eos = ModifiedNASG()

        # Initial conditions
        self.P_L = P_L
        self.P_R = P_R
        self.T_L = T_L
        self.T_R = T_R
        self.V_L = 0.0  # Initial velocity left
        self.V_R = 0.0  # Initial velocity right
        self.x0 = x0
        self.L = L

        # Calculate initial densities
        self.rho_L = self.eos.density(P_L, T_L)
        self.rho_R = self.eos.density(P_R, T_R)

        # Calculate initial sound speeds
        self.a_L = self.eos.sound_speed(P_L, T_L)
        self.a_R = self.eos.sound_speed(P_R, T_R)

        # Stiffened quantities
        self.P_bar_L = self.eos.stiffened_pressure(P_L)
        self.P_bar_R = self.eos.stiffened_pressure(P_R)
        self.rho_bar_L = self.eos.stiffened_density(self.rho_L)
        self.rho_bar_R = self.eos.stiffened_density(self.rho_R)

        # Solution variables (to be computed)
        self.P1 = None
        self.P2 = None
        self.V1 = None
        self.V2 = None
        self.rho1 = None
        self.rho2 = None
        self.T1 = None
        self.T2 = None
        self.V_shock = None

    def solve_shock_pressure_ratio(self):
        """
        Solve for shock pressure ratio using Regula-Falsi method
        Equation (10) from the paper
        """
        gamma = self.eos.gamma

        def pressure_ratio_equation(P1_Pr):
            """Implicit equation for shock pressure ratio"""
            term1 = self.P_bar_L / self.P_bar_R
            term2_num = (gamma - 1) * (self.a_R / self.a_L) * (P1_Pr - 1)
            term2_den = np.sqrt(2 * gamma * (2 * gamma + (gamma + 1) * (P1_Pr - 1)))
            term2 = term2_num / term2_den
            term3_exp = -2 * gamma / (gamma - 1)

            lhs = term1
            rhs = P1_Pr * (1 - term2) ** term3_exp

            return lhs - rhs

        # Initial guess
        P1_Pr_guess = self.P_L / self.P_R

        # Solve using fsolve
        P1_Pr_solution = fsolve(pressure_ratio_equation, P1_Pr_guess)[0]

        # Calculate stiffened pressure in region 1
        P_bar_1 = P1_Pr_solution * self.P_bar_R

        # Convert back to absolute pressure
        self.P1 = P_bar_1 - self.eos.P_inf

        return P1_Pr_solution

    def calculate_shock_speed(self):
        """Calculate shock speed (Equation 11)"""
        gamma = self.eos.gamma
        P_bar_1 = self.eos.stiffened_pressure(self.P1)
        P1_Pr = P_bar_1 / self.P_bar_R

        term1 = (gamma + 1) / (2 * gamma)
        term2 = (gamma - 1) / (2 * gamma)

        self.V_shock = self.a_R * np.sqrt(term1 * P1_Pr + term2)

        return self.V_shock

    def calculate_region1_velocity(self):
        """Calculate velocity in region 1 (upstream of shock) - Equation 12"""
        gamma = self.eos.gamma
        P_bar_1 = self.eos.stiffened_pressure(self.P1)
        P1_Pr = P_bar_1 / self.P_bar_R

        num = self.a_R * (P1_Pr - 1)
        den_term1 = (2 * gamma) / (gamma + 1)
        den_term2 = P1_Pr + (gamma - 1) / (gamma + 1)
        den = gamma * np.sqrt(den_term1 / den_term2)

        self.V1 = num / den

        return self.V1

    def calculate_region1_density(self):
        """Calculate density in region 1 - Equation 13, 14"""
        gamma = self.eos.gamma
        P_bar_1 = self.eos.stiffened_pressure(self.P1)
        P1_Pr = P_bar_1 / self.P_bar_R

        # Stiffened density
        num = ((gamma + 1) / (gamma - 1)) * P1_Pr + 1
        den = ((gamma + 1) / (gamma - 1)) + P1_Pr
        rho_bar_1 = self.rho_bar_R * (num / den)

        # Convert to absolute density
        self.rho1 = rho_bar_1 / (1 + self.eos.b * rho_bar_1)

        return self.rho1

    def calculate_region2_properties(self):
        """Calculate properties in region 2 - Equations 15, 16"""
        # Velocity and pressure same as region 1
        self.V2 = self.V1
        self.P2 = self.P1

        # Calculate density using isentropic relation
        gamma = self.eos.gamma
        P_bar_2 = self.eos.stiffened_pressure(self.P2)

        rho_bar_2 = self.rho_bar_L * (P_bar_2 / self.P_bar_L) ** (1 / gamma)

        # Convert to absolute density
        self.rho2 = rho_bar_2 / (1 + self.eos.b * rho_bar_2)

        return self.rho2

    def calculate_temperatures(self):
        """Calculate temperatures using EOS - Equation 27"""
        # Region 1
        v1 = 1.0 / self.rho1
        self.T1 = (self.P1 + self.eos.P_inf) * (v1 - self.eos.b) / ((self.eos.gamma - 1) * self.eos.cv)

        # Region 2
        v2 = 1.0 / self.rho2
        self.T2 = (self.P2 + self.eos.P_inf) * (v2 - self.eos.b) / ((self.eos.gamma - 1) * self.eos.cv)

        return self.T1, self.T2

    def expansion_fan_properties(self, x, t):
        """
        Calculate properties inside expansion fan
        Equations 20-26
        """
        gamma = self.eos.gamma

        # Sound speed in expansion fan (Eq. 20)
        factor1 = 2 * self.a_L / (1 + gamma)
        factor2 = (1 - gamma) / (1 + gamma)
        factor3 = (x - self.x0) / t
        a_E = factor1 * (1 + factor2 * factor3)

        # Stiffened pressure in expansion fan (Eq. 21)
        P_bar_E = self.P_bar_L * (a_E / self.a_L) ** (2 * gamma / (gamma - 1))
        P_E = P_bar_E - self.eos.P_inf

        # Stiffened density in expansion fan (Eq. 23)
        rho_bar_E = self.rho_bar_L * (P_bar_E / self.P_bar_L) ** (1 / gamma)
        rho_E = rho_bar_E / (1 + self.eos.b * rho_bar_E)

        # Velocity in expansion fan (Eq. 26)
        V_E = (2 / (gamma - 1)) * (self.a_L - a_E)

        # Temperature in expansion fan
        v_E = 1.0 / rho_E
        T_E = (P_E + self.eos.P_inf) * (v_E - self.eos.b) / ((gamma - 1) * self.eos.cv)

        return V_E, P_E, rho_E, T_E, a_E

    def solve(self):
        """Solve the complete Riemann problem"""
        # Step 1: Solve for shock pressure ratio
        print("Solving for shock pressure ratio...")
        P1_Pr = self.solve_shock_pressure_ratio()
        print(f"  P1/PR = {P1_Pr:.4f}")
        print(f"  P1 = {self.P1/1e6:.4f} MPa")

        # Step 2: Calculate shock speed
        print("\nCalculating shock speed...")
        V_s = self.calculate_shock_speed()
        print(f"  Shock speed = {V_s:.2f} m/s")

        # Step 3: Calculate region 1 properties
        print("\nCalculating region 1 properties...")
        V1 = self.calculate_region1_velocity()
        rho1 = self.calculate_region1_density()
        print(f"  Velocity = {V1:.4f} m/s")
        print(f"  Density = {rho1:.2f} kg/m³")

        # Step 4: Calculate region 2 properties
        print("\nCalculating region 2 properties...")
        rho2 = self.calculate_region2_properties()
        print(f"  Velocity = {self.V2:.4f} m/s")
        print(f"  Pressure = {self.P2/1e6:.4f} MPa")
        print(f"  Density = {rho2:.2f} kg/m³")

        # Step 5: Calculate temperatures
        print("\nCalculating temperatures...")
        T1, T2 = self.calculate_temperatures()
        print(f"  T1 = {T1:.2f} K")
        print(f"  T2 = {T2:.2f} K")

        print("\nSolution complete!")

    def get_solution(self, x, t):
        """
        Get solution at position x and time t
        Equation 28 - piece-wise solution

        Returns:
        --------
        V, P, rho, T, a : flow variables at (x, t)
        """
        # Calculate zone boundaries (Eq. 17)
        x_A = self.x0 - self.a_L * t

        a_2 = self.eos.sound_speed(self.P2, self.T2)
        x_B = self.x0 + (self.V2 - a_2) * t

        x_C = self.x0 + self.V2 * t
        x_D = self.x0 + self.V_shock * t

        # Determine which zone x is in
        if x < x_A:
            # Zone L (left undisturbed)
            return self.V_L, self.P_L, self.rho_L, self.T_L, self.a_L

        elif x_A <= x < x_B:
            # Zone E (expansion fan)
            return self.expansion_fan_properties(x, t)

        elif x_B <= x < x_C:
            # Zone 2
            a_2 = self.eos.sound_speed(self.P2, self.T2)
            return self.V2, self.P2, self.rho2, self.T2, a_2

        elif x_C <= x < x_D:
            # Zone 1
            a_1 = self.eos.sound_speed(self.P1, self.T1)
            return self.V1, self.P1, self.rho1, self.T1, a_1

        else:
            # Zone R (right undisturbed)
            return self.V_R, self.P_R, self.rho_R, self.T_R, self.a_R

    def plot_solution(self, t, N=1000):
        """
        Plot solution profiles at time t

        Parameters:
        -----------
        t : float
            Time to plot solution (seconds)
        N : int
            Number of spatial points
        """
        x = np.linspace(0, self.L, N)

        V = np.zeros(N)
        P = np.zeros(N)
        rho = np.zeros(N)
        T = np.zeros(N)
        a = np.zeros(N)

        for i, xi in enumerate(x):
            V[i], P[i], rho[i], T[i], a[i] = self.get_solution(xi, t)

        # Create subplots
        fig, axs = plt.subplots(2, 2, figsize=(14, 10))

        # Velocity
        axs[0, 0].plot(x, V, 'b-', linewidth=2)
        axs[0, 0].set_xlabel('X (m)', fontsize=12)
        axs[0, 0].set_ylabel('Velocity (m/s)', fontsize=12)
        axs[0, 0].set_title(f'Velocity Profile at t = {t*1e6:.0f} μs', fontsize=14)
        axs[0, 0].grid(True, alpha=0.3)

        # Pressure
        axs[0, 1].plot(x, P/1e6, 'r-', linewidth=2)
        axs[0, 1].set_xlabel('X (m)', fontsize=12)
        axs[0, 1].set_ylabel('Pressure (MPa)', fontsize=12)
        axs[0, 1].set_title(f'Pressure Profile at t = {t*1e6:.0f} μs', fontsize=14)
        axs[0, 1].grid(True, alpha=0.3)

        # Density
        axs[1, 0].plot(x, rho, 'g-', linewidth=2)
        axs[1, 0].set_xlabel('X (m)', fontsize=12)
        axs[1, 0].set_ylabel('Density (kg/m³)', fontsize=12)
        axs[1, 0].set_title(f'Density Profile at t = {t*1e6:.0f} μs', fontsize=14)
        axs[1, 0].grid(True, alpha=0.3)

        # Temperature
        axs[1, 1].plot(x, T, 'm-', linewidth=2)
        axs[1, 1].set_xlabel('X (m)', fontsize=12)
        axs[1, 1].set_ylabel('Temperature (K)', fontsize=12)
        axs[1, 1].set_title(f'Temperature Profile at t = {t*1e6:.0f} μs', fontsize=14)
        axs[1, 1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(f'water_shock_tube_t{t*1e6:.0f}us.png', dpi=300, bbox_inches='tight')
        plt.show()

        return x, V, P, rho, T



In [ ]:
# ==============================================================================
# Example Usage
# ==============================================================================

if __name__ == "__main__":
    print("="*70)
    print("EXACT RIEMANN SOLVER FOR WATER SHOCK TUBE")
    print("Modified NASG Equation of State")
    print("="*70)

    # Example 1: Case from the paper (PL = 10 MPa)
    print("\n" + "="*70)
    print("CASE 1: PL = 10 MPa, PR = 0.1 MPa, T = 300 K")
    print("="*70)

    solver1 = WaterShockTubeSolver(
        P_L=1e6,      # 10 MPa
        P_R=0.1e6,     # 0.1 MPa
        T_L=300,       # 300 K
        T_R=300,       # 300 K
        x0=2.0,        # Diaphragm at 2.0 m
        L=3.0          # Total length 3.0 m
    )

    solver1.solve()

    # Plot at t = 200 microseconds
    t = 400e-6  # 200 μs
    x1, V1, P1, rho1, T1 = solver1.plot_solution(t)

    # Example 2: Higher pressure case (PL = 100 MPa)
    print("\n" + "="*70)
    print("CASE 2: PL = 100 MPa, PR = 0.1 MPa, T = 300 K")
    print("="*70)



    print("\n" + "="*70)
    print("ALL CASES COMPLETED SUCCESSFULLY!")
    print("="*70)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load data from your file (assumes whitespace separated, 4 columns: x, p, rho, T)
data = np.loadtxt('postProcessing/sets/0.0004/data_p_rho_T.xy')
data2 = np.loadtxt('GS/sets/0.0004/data_p_rho_T.xy')

# Unpack columns for clarity
x_numerical = data[:, 0]
p_numerical = data[:, 1]
rho_numerical = data[:, 2]
T_numerical = data[:, 3]
# Unpack columns for clarity
x_GS = data2[:, 0]
p_GS = data2[:, 1]
rho_GS = data2[:, 2]
T_GS = data2[:, 3]

# pExact = np.loadtxt('exactSolution/p_exact.dat')
# x_pExact = pExact[:, 0]
# p_pExact = pExact[:, 1]

# TExact = np.loadtxt('exactSolution/T_exact.dat')
# x_TExact = TExact[:, 0]
# T_TExact = TExact[:, 1]

# rhoExact = np.loadtxt('exactSolution/rho_exact.dat')
# x_rhoExact = rhoExact[:, 0]
# rho_rhoExact = rhoExact[:, 1]
# x1, V1, P1, rho1, T1
# pExact = np.loadtxt('exactSolution/p_exact.dat')
x_pExact = x1
p_pExact = P1

# TExact = np.loadtxt('exactSolution/T_exact.dat')
x_TExact = x1
T_TExact = T1

x_rhoExact = x1
rho_rhoExact = rho1

# Plot Pressure
plt.figure(figsize=(8, 5))
plt.plot(x_numerical, p_GS, '-', label='GS: Pressure')
plt.plot(x_GS, p_numerical, '-', label='LS-VK: Pressure')
plt.plot(x1, P1, '-', label='Exact: Pressure')
plt.xlabel('Position')
plt.ylabel('Pressure')
plt.legend()
plt.title('Pressure Comparison')
plt.show()

# Plot Density
plt.figure(figsize=(8, 5))
plt.plot(x_numerical, rho_numerical, '-', label='Numerical: Density')
plt.plot(x1, rho1, '-', label='Exact: Density')
plt.xlabel('Position')
plt.ylabel('Density')
plt.legend()
plt.title('Density Comparison')
plt.show()

# Plot Temperature
plt.figure(figsize=(8, 5))
plt.plot(x_GS, T_GS, '-', label='GS: Temperature')
plt.plot(x_numerical, T_numerical, '-', label='Numerical: Temperature')
plt.plot(x1, T1, '-', label='Exact: Temperature')
plt.xlabel('Position')
plt.ylabel('Temperature')
plt.legend()
plt.title('Temperature Comparison')
plt.show()

# You can similarly plot velocity (u) if your file or arrays support it.